# Real-time video stabilization

Causal stabilization — the pipeline never looks at future frames, so it runs on
a live stream with no added latency. Built for a Jetson Orin Nano, with a
synchronized stereo pair as the next step.

Everything lives in cells, so nothing is lost when the Colab session ends.
Sections A–E are the main run; the appendix keeps the experiments that led to
the current settings.

| section | what it does |
|---|---|
| A. Setup | definitions, run once |
| B. Data | three benchmark clips + settings |
| C. Run | process all clips with one config |
| D. Results | table, charts, videos |
| E. Deployment | what changes on the Jetson |
| Appendix | parameter sweeps and diagnostics |


---
## A. Setup

Run these three once per session.


### A1. Core stabilizer

This is the only part that ships to the Jetson. Everything below it exists to test it.


In [ ]:
# Core stabilizer: corner-based budget, soft limiting,
# grid-distributed features, confidence weighting.
"""
Causal video stabilization for live streams.

Design notes
------------
1) No future frames. process() only ever sees what has already arrived.
   Trajectory smoothing is an EMA (IIR), not a sliding window, so the
   output frame is produced immediately with zero added latency.
2) Sources are pluggable. VideoFileSource for offline work today,
   CsiCameraSource on the Jetson tomorrow; nothing else changes.
3) Built with stereo in mind. Every frame carries a timestamp, a camera
   id and the warp that was applied. process() also accepts an
   external_correction so a future sync layer can force both cameras to
   use the *same* correction -- stabilizing two cameras independently
   would break the stereo geometry.
4) Speed. Motion is estimated on a downscaled grayscale frame; rotation,
   translation and the crop-zoom all go into a single warpAffine.
"""
from __future__ import annotations
import math, os, time
from dataclasses import dataclass
from typing import Iterator, Optional, Tuple, List
import cv2
import numpy as np


@dataclass
class Frame:
    image: np.ndarray
    timestamp: float
    index: int
    cam_id: int = 0


@dataclass
class Motion:
    dx: float = 0.0
    dy: float = 0.0
    da: float = 0.0          # radians
    valid: bool = False
    n_inliers: int = 0
    def as_array(self):
        return np.array([self.dx, self.dy, self.da], dtype=np.float64)


@dataclass
class StabilizedFrame:
    image: np.ndarray
    timestamp: float
    index: int
    cam_id: int
    correction: np.ndarray
    motion: Motion
    warp: np.ndarray
    n_tracks: int
    proc_ms: float


@dataclass
class StabConfig:
    proc_width: int = 480          # motion-estimation width; biggest speed lever
    max_corners: int = 200
    quality_level: float = 0.01
    min_distance: int = 12
    block_size: int = 3
    redetect_interval: int = 15
    min_tracks: int = 60
    lk_win: int = 21
    lk_levels: int = 3
    use_fb_check: bool = True      # forward-backward check (~20% slower)
    fb_threshold: float = 1.0
    ransac_thresh: float = 3.0
    min_inliers: int = 12
    # Defaults come from the benchmark sweep (see the appendix notebook).
    # adaptive=True was measured to fire almost constantly and wreck the
    # smoothing, so it is off by default. It may still help on footage with
    # fast sustained panning -- check visually before turning it back on.
    smooth_alpha: float = 0.90     # 0.85 = responsive, 0.97 = very smooth
    adaptive: bool = False
    crop_ratio: float = 0.12       # crop per side; zoom = 1/(1-2r)
    grid: int = 4                  # feature distribution grid
    conf_inliers: int = 40         # inlier count for full confidence
    windup_leak: float = 0.25      # anti-windup leak rate
    max_angle_deg: float = 3.0
    safety: float = 0.95           # unused since the corner check replaced it
    num_threads: int = 0
    interpolation: int = cv2.INTER_LINEAR
    border_mode: int = cv2.BORDER_REPLICATE


class FrameSource:
    def read(self) -> Optional[Frame]:
        raise NotImplementedError
    def release(self): pass
    def __iter__(self):
        while True:
            f = self.read()
            if f is None: break
            yield f


class VideoFileSource(FrameSource):
    """Reads a file but behaves like a camera: no seeking, no look-ahead."""
    def __init__(self, path, cam_id=0, realtime=False):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dosya bulunamadi: {path}")
        self.cap = cv2.VideoCapture(path)
        if not self.cap.isOpened():
            raise IOError(f"Video acilamadi (codec?): {path}")
        self.cam_id, self.realtime, self.index = cam_id, realtime, -1
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self._t0 = time.monotonic()

    def read(self):
        ok, img = self.cap.read()
        if not ok: return None
        self.index += 1
        ts = self.index / self.fps
        if self.realtime:
            d = (self._t0 + ts) - time.monotonic()
            if d > 0: time.sleep(d)
        return Frame(img, ts, self.index, self.cam_id)

    def release(self): self.cap.release()


class CsiCameraSource(FrameSource):
    """Jetson CSI camera. Won't run off-device; here so deployment is a no-op."""
    GST = ("nvarguscamerasrc sensor-id={sid} ! "
           "video/x-raw(memory:NVMM), width={w}, height={h}, framerate={fps}/1 ! "
           "nvvidconv ! video/x-raw, format=BGRx ! "
           "videoconvert ! video/x-raw, format=BGR ! "
           "appsink drop=true max-buffers=1 sync=false")

    def __init__(self, sensor_id=0, width=1920, height=1080, fps=30, cam_id=None):
        self.cap = cv2.VideoCapture(
            self.GST.format(sid=sensor_id, w=width, h=height, fps=fps),
            cv2.CAP_GSTREAMER)
        if not self.cap.isOpened():
            raise IOError(f"CSI kamera acilamadi (sensor-id={sensor_id})")
        self.cam_id = sensor_id if cam_id is None else cam_id
        self.index = -1

    def read(self):
        ok, img = self.cap.read()
        if not ok: return None
        self.index += 1
        return Frame(img, time.monotonic(), self.index, self.cam_id)

    def release(self): self.cap.release()


class OnlineStabilizer:
    """
    frame -> downscale + gray -> LK tracking -> partial affine (RANSAC)
          -> cumulative trajectory -> EMA -> correction -> limit -> one warp
    """
    def __init__(self, cfg: StabConfig = StabConfig(), cam_id: int = 0):
        self.cfg, self.cam_id = cfg, cam_id
        if cfg.num_threads > 0: cv2.setNumThreads(cfg.num_threads)
        cv2.setUseOptimized(True)
        self._lk = dict(winSize=(cfg.lk_win, cfg.lk_win), maxLevel=cfg.lk_levels,
                        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))
        self.reset()

    def reset(self):
        self.prev_gray = None
        self.prev_pts = None
        self.traj = np.zeros(3)
        self.smooth = np.zeros(3)
        self.frame_count = 0
        self.scale = 1.0
        self.size = None
        self._center = (0.0, 0.0)
        self._zoom = 1.0 / (1.0 - 2.0 * self.cfg.crop_ratio)

    def _prepare(self, image):
        h, w = image.shape[:2]
        if self.size is None:
            self.size = (w, h)
            self.scale = min(1.0, self.cfg.proc_width / float(w))
            self._center = (w / 2.0, h / 2.0)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        if self.scale < 1.0:
            gray = cv2.resize(gray, None, fx=self.scale, fy=self.scale,
                              interpolation=cv2.INTER_AREA)
        return gray

    def _detect(self, gray, keep=None):
        """Detect features with a per-cell quota.

        goodFeaturesToTrack ranks corners globally, so one high-texture
        region -- a passing car, a close-up object -- can take most of the
        budget and drag RANSAC onto *its* motion instead of the camera's.
        Passing `keep` preserves surviving tracks and only fills the gaps."""
        cfg = self.cfg
        h, w = gray.shape
        gy = gx = cfg.grid
        per = max(4, cfg.max_corners // (gx*gy))
        mask = None
        if keep is not None and len(keep):
            mask = np.full((h, w), 255, np.uint8)
            for x, y in keep.reshape(-1, 2).astype(int):
                cv2.circle(mask, (x, y), cfg.min_distance, 0, -1)
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1 = i*h//gy, (i+1)*h//gy
                x0, x1 = j*w//gx, (j+1)*w//gx
                sub = None if mask is None else mask[y0:y1, x0:x1]
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], per,
                        cfg.quality_level, cfg.min_distance,
                        mask=sub, blockSize=cfg.block_size)
                if p is not None:
                    p[:,0,0] += x0; p[:,0,1] += y0
                    pts.append(p)
        new = np.vstack(pts) if pts else None
        if keep is None or not len(keep):
            return new
        return keep if new is None else np.vstack([keep, new]).astype(np.float32)

    def _track(self, pg, cg, p0):
        p1, st, _ = cv2.calcOpticalFlowPyrLK(pg, cg, p0, None, **self._lk)
        if p1 is None: return None, None
        st = st.reshape(-1).astype(bool)
        if self.cfg.use_fb_check and st.any():
            pb, st2, _ = cv2.calcOpticalFlowPyrLK(cg, pg, p1, None, **self._lk)
            if pb is not None:
                err = np.linalg.norm(p0.reshape(-1, 2) - pb.reshape(-1, 2), axis=1)
                st &= st2.reshape(-1).astype(bool) & (err < self.cfg.fb_threshold)
        if st.sum() < 6: return None, None
        return p0[st], p1[st]

    def _estimate(self, p0, p1) -> Motion:
        M, inl = cv2.estimateAffinePartial2D(
            p0, p1, method=cv2.RANSAC, ransacReprojThreshold=self.cfg.ransac_thresh,
            maxIters=500, confidence=0.99, refineIters=10)
        if M is None: return Motion()
        n = int(inl.sum()) if inl is not None else 0
        if n < self.cfg.min_inliers: return Motion(n_inliers=n)
        s = 1.0 / self.scale
        return Motion(float(M[0, 2]) * s, float(M[1, 2]) * s,
                      float(math.atan2(M[1, 0], M[0, 0])), True, n)

    def _fits(self, M, W, H):
        """Do all four output corners land inside the source frame?

        Checks rotation, zoom and translation together. Limiting each axis
        separately missed the extra margin rotation eats at the corners --
        at crop=0.10 and 6 degrees the frame overflowed by ~53 px and
        BORDER_REPLICATE smeared the edges."""
        Mi = cv2.invertAffineTransform(M)
        dst = np.array([[0,0],[W,0],[W,H],[0,H]], np.float32).reshape(-1,1,2)
        q = cv2.transform(dst, Mi).reshape(-1,2)
        return (q[:,0].min() >= 0 and q[:,1].min() >= 0
                and q[:,0].max() <= W and q[:,1].max() <= H)

    def _soft_limit(self, corr, mx, my, ma, knee=0.6):
        """Squash toward the limit with tanh past a knee instead of clipping.

        np.clip has a discontinuous derivative, so every time the correction
        hit the wall its velocity dropped to zero and the picture snapped --
        the most visible artifact this stabilizer produced."""
        lim = (mx, my, ma)
        for i in range(3):
            r = abs(corr[i]) / max(lim[i], 1e-9)
            if r > knee:
                t = (r - knee) / (1 - knee)
                corr[i] *= (knee + (1 - knee) * math.tanh(t)) / r
        return corr

    def _smooth_step(self, mx, my):
        cfg = self.cfg
        a = cfg.smooth_alpha
        self.smooth = a * self.smooth + (1 - a) * self.traj
        corr = self.smooth - self.traj
        if cfg.adaptive:
            r = max(abs(corr[0]) / max(mx, 1e-6), abs(corr[1]) / max(my, 1e-6))
            if r > 0.7:   # track the camera faster as we approach the limit
                ae = a - (a - 0.60) * min((r - 0.7) / 0.3, 1.0)
                self.smooth = ae * self.smooth + (1 - ae) * self.traj
                corr = self.smooth - self.traj
        ma = math.radians(cfg.max_angle_deg)
        corr = self._soft_limit(corr, mx, my, ma)
        # anti-windup: leak toward the limited value instead of snapping to it
        self.smooth += cfg.windup_leak * ((self.traj + corr) - self.smooth)
        return corr

    def _warp_matrix(self, corr):
        # Sign: estimateAffinePartial2D and getRotationMatrix2D disagree on
        # rotation direction, hence the minus. Without it the correction
        # roughly doubled the rotation instead of removing it.
        M = cv2.getRotationMatrix2D(self._center, -math.degrees(corr[2]), self._zoom)
        # The crop-zoom magnifies the image by z, so it magnifies the motion
        # inside it too. Without scaling the correction by z, a fraction
        # (z-1)/z of the shake survives -- 32% at crop=0.12.
        M[0, 2] += corr[0] * self._zoom
        M[1, 2] += corr[1] * self._zoom
        return M

    def process(self, frame: Frame, external_correction=None) -> StabilizedFrame:
        t0 = time.perf_counter()
        cfg = self.cfg
        gray = self._prepare(frame.image)
        W, H = self.size
        mx = cfg.crop_ratio * W * cfg.safety
        my = cfg.crop_ratio * H * cfg.safety

        motion = Motion()
        need = (self.prev_pts is None or len(self.prev_pts) < cfg.min_tracks
                or self.frame_count % cfg.redetect_interval == 0)
        if self.prev_gray is not None:
            if need: self.prev_pts = self._detect(self.prev_gray, keep=self.prev_pts)
            if self.prev_pts is not None and len(self.prev_pts) >= 6:
                p0, p1 = self._track(self.prev_gray, gray, self.prev_pts)
                if p0 is not None:
                    motion = self._estimate(p0, p1)
                    self.prev_pts = p1.reshape(-1, 1, 2).astype(np.float32)
                else:
                    self.prev_pts = None
            else:
                self.prev_pts = None

        if motion.valid:
    # Weight by inlier count rather than treating motion as all-or-nothing.
            # Assuming "the camera stopped" on a blurry frame made the
            # trajectory jump once tracking recovered.
            wgt = min(1.0, motion.n_inliers / float(cfg.conf_inliers))
            self.traj += wgt * motion.as_array()
        corr = self._smooth_step(mx, my)
        if external_correction is not None:
            corr = np.asarray(external_correction, dtype=np.float64)
            self.smooth = self.traj + corr

        M = self._warp_matrix(corr)
        for _ in range(6):                      # shrink until it fits
            if self._fits(M, W, H): break
            corr *= 0.85
            M = self._warp_matrix(corr)
        out = cv2.warpAffine(frame.image, M, (W, H),
                             flags=cfg.interpolation, borderMode=cfg.border_mode)
        self.prev_gray = gray
        self.frame_count += 1
        return StabilizedFrame(out, frame.timestamp, frame.index, frame.cam_id,
                               corr, motion, M,
                               0 if self.prev_pts is None else len(self.prev_pts),
                               (time.perf_counter() - t0) * 1000.0)


class CameraPipeline:
    """One source + one stabilizer. The sync layer will hold two of these."""
    def __init__(self, source, cfg=StabConfig(), cam_id=0):
        self.source, self.cam_id = source, cam_id
        self.stab = OnlineStabilizer(cfg, cam_id=cam_id)
    def next(self, external_correction=None):
        f = self.source.read()
        if f is None: return None
        f.cam_id = self.cam_id
        return self.stab.process(f, external_correction=external_correction)
    def release(self): self.source.release()


print("Core loaded.")

### A2. Measurement and benchmark helpers


In [ ]:
"""
Offline evaluation helpers. None of this ships to the Jetson -- it exists to
measure whether the stabilizer is doing its job and to pick parameters.

Two metrics live here and they answer different questions:

  measure_motion / compare_motion
      Mean frame-to-frame displacement of the output video. Includes the
      deliberate camera motion we are supposed to *keep*, so on pan-heavy
      footage it can never approach 100%. Useful, but easy to misread.

  high-frequency suppression (see hf_score)
      Removes the low-frequency component first, then compares what is left.
      This is the one that tracks what the eye calls "shaky", and it is the
      metric parameter choices should be made against.
"""

import math
import os
import subprocess
from typing import Optional
from urllib.request import urlretrieve

import cv2
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


SAMPLES = {
    # MeshFlow (Liu et al., ECCV 2016) sample clip -- academic source, fast host
    "meshflow": ("https://raw.githubusercontent.com/sudheerachary/"
                 "Mesh-Flow-Video-Stabilization/master/data/shaky-5.avi"),
    "meshflow_small": ("https://raw.githubusercontent.com/sudheerachary/"
                       "Mesh-Flow-Video-Stabilization/master/data/small-shaky-5.avi"),
    "ostrich": "https://s3.amazonaws.com/python-vidstab/ostrich.mp4",
    "thrasher": "https://s3.amazonaws.com/python-vidstab/thrasher.mp4",
}


def download_sample(name="meshflow", dest="samples"):
    if name not in SAMPLES:
        raise ValueError(f"Unknown sample: {name}. Options: {list(SAMPLES)}")
    os.makedirs(dest, exist_ok=True)
    ext = os.path.splitext(SAMPLES[name])[1] or ".mp4"
    path = os.path.join(dest, f"{name}{ext}")
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        print(f"[cached] {path}")
        return path
    print(f"[downloading] {SAMPLES[name]}")
    urlretrieve(SAMPLES[name], path)
    c = cv2.VideoCapture(path)
    print(f"[ready] {path}  {int(c.get(3))}x{int(c.get(4))} "
          f"{c.get(5):.1f}fps {int(c.get(7))} frames "
          f"({os.path.getsize(path)/1e6:.1f} MB)")
    c.release()
    return path


def add_shake(src_path, out_path=None, amp=15.0, rot=2.0, noise=0.4,
              max_frames=300, seed=0):
    """Inject shake of known strength and save the ground truth alongside.

    The ground truth is stored as the affine parameters of the injected
    matrix, not as the raw jx/jy offsets: rotation about the centre produces
    extra translation away from the centre, and the estimator measures that
    inside tx/ty. Storing jx/jy would mean the two sides measure different
    things and the suppression numbers come out nonsensical.
    """
    rng = np.random.default_rng(seed)
    if out_path is None:
        b, e = os.path.splitext(src_path)
        out_path = f"{b}_shake_a{amp:g}_r{rot:g}{e}"   # params in the filename
    if os.path.exists(out_path) and os.path.exists(out_path + ".gt.npy"):
        print(f"[cached] {out_path}")
        return out_path, out_path + ".gt.npy"

    src = VideoFileSource(src_path)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W, H))
    gt, i = [], 0
    while True:
        f = src.read()
        if f is None or (max_frames and i >= max_frames):
            break
        jx = amp * math.sin(i*1.7) + amp*noise*rng.standard_normal()
        jy = amp*0.8*math.sin(i*2.3) + amp*noise*rng.standard_normal()
        ja = rot * math.sin(i*1.1) + rot*noise*rng.standard_normal()
        M = cv2.getRotationMatrix2D((W/2., H/2.), ja, 1.0)
        M[0, 2] += jx
        M[1, 2] += jy
        w.write(cv2.warpAffine(f.image, M, (W, H), borderMode=cv2.BORDER_REPLICATE))
        gt.append([float(M[0, 2]), float(M[1, 2]),
                   float(math.atan2(M[1, 0], M[0, 0]))])
        i += 1
    w.release()
    src.release()
    np.save(out_path + ".gt.npy", np.array(gt))
    print(f"[generated] {out_path}  ({i} frames, amp={amp}, rot={rot})")
    return out_path, out_path + ".gt.npy"


def stabilize_clip(path, cfg=None, out_dir="out", max_frames=None,
                   side_by_side=True, to_h264=True, quiet=False):
    """Run a clip through the stabilizer one frame at a time, as if live."""
    cfg = cfg or StabConfig()
    os.makedirs(out_dir, exist_ok=True)
    name = os.path.splitext(os.path.basename(path))[0]
    tmp = os.path.join(out_dir, f"{name}_tmp.mp4")
    final = os.path.join(out_dir, f"{name}_cmp.mp4" if side_by_side
                         else f"{name}_stab.mp4")
    src = VideoFileSource(path)
    stab = OnlineStabilizer(cfg)
    W, H = src.width, src.height
    wr = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), src.fps,
                         (W*2 if side_by_side else W, H))
    times, raw_j, stab_j, corrs = [], [], [], []
    prev = np.zeros(3)
    n = 0
    while True:
        f = src.read()
        if f is None:
            break
        sf = stab.process(f)
        times.append(sf.proc_ms)
        corrs.append(sf.correction.copy())
        if sf.motion.valid:
            raw_j.append(math.hypot(sf.motion.dx, sf.motion.dy))
            z = stab._zoom            # the zoom magnifies output motion by z
            r = z * (sf.motion.as_array() + (sf.correction - prev))
            stab_j.append(math.hypot(r[0], r[1]))
        prev = sf.correction.copy()
        wr.write(np.hstack([f.image, sf.image]) if side_by_side else sf.image)
        n += 1
        if max_frames and n >= max_frames:
            break
    wr.release()
    src.release()

    if to_h264 and subprocess.call(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                                    "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                                    "-crf", "24", final]) == 0:
        os.remove(tmp)
    else:
        os.replace(tmp, final)

    t = np.array(times) if times else np.array([0.0])
    res = {"clip": os.path.basename(path), "res": f"{W}x{H}", "frames": n,
           "ms": float(t.mean()), "ms_p95": float(np.percentile(t, 95)),
           "fps": float(1000/t.mean()) if t.mean() else 0.0,
           "jit_raw": float(np.mean(raw_j)) if raw_j else 0.0,
           "jit_stab": float(np.mean(stab_j)) if stab_j else 0.0,
           "corrections": np.array(corrs), "times": t, "cfg": cfg, "out": final}
    if not quiet:
        red = (1 - res["jit_stab"]/res["jit_raw"])*100 if res["jit_raw"] else 0
        print(f"{res['res']} | {n} frames | {res['ms']:.2f} ms "
              f"(p95 {res['ms_p95']:.2f}) -> {res['fps']:.1f} FPS")
        print(f"jitter: {res['jit_raw']:.2f} -> {res['jit_stab']:.2f} px/frame "
              f"({red:.1f}% reduction)")
    return res


# --------------------------------------------------------------------------
# metrics
# --------------------------------------------------------------------------

def measure_motion(path, max_frames=400, half=None):
    """Frame-to-frame motion of a video on its own.

    half='right' measures only the stabilized half of a side-by-side output;
    without it half the frame is the raw footage and the number is garbage.
    """
    stab = OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
    src = VideoFileSource(path)
    m, i = [], 0
    while i < max_frames:
        f = src.read()
        if f is None:
            break
        if half:
            w = f.image.shape[1] // 2
            f.image = f.image[:, :w] if half == "left" else f.image[:, w:]
        sf = stab.process(f)
        if sf.motion.valid:
            m.append(sf.motion.as_array())
        i += 1
    src.release()
    if not m:
        return {"dxy": 0.0, "da_std": 0.0, "frames": 0}
    m = np.array(m)
    return {"dxy": float(np.mean(np.hypot(m[:, 0], m[:, 1]))),
            "da_std": float(np.degrees(np.std(m[:, 2]))), "frames": len(m)}


def compare_motion(in_path, out_path, max_frames=400, out_half=None, out_zoom=1.0):
    """Input vs output motion.

    out_zoom must be 1/(1-2*crop_ratio). The output is zoomed, so everything
    in it -- including the camera motion we deliberately keep -- is magnified
    by z. Skip the division and results get worse as crop_ratio goes up, which
    is an artifact, not a real effect. Rotation is unaffected by zoom.
    """
    a = measure_motion(in_path, max_frames)
    b = measure_motion(out_path, max_frames, half=out_half)
    b = {**b, "dxy": b["dxy"] / out_zoom}
    dr = (1 - b["dxy"]/a["dxy"])*100 if a["dxy"] else 0.0
    ar = (1 - b["da_std"]/a["da_std"])*100 if a["da_std"] else 0.0
    print(f"{'':8s} {'translation':>13s} {'rotation':>12s}")
    print(f"{'input':8s} {a['dxy']:10.2f} px {a['da_std']:9.2f} deg")
    print(f"{'output':8s} {b['dxy']:10.2f} px {b['da_std']:9.2f} deg")
    print(f"{'reduced':8s} {dr:10.1f} %  {ar:9.1f} %")
    return {"in": a, "out": b, "dxy_red": dr, "da_red": ar}


def _trajectory(path, n=300):
    st = OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
    src = VideoFileSource(path)
    m, i = [], 0
    while i < n:
        f = src.read()
        if f is None:
            break
        sf = st.process(f)
        if sf.motion.valid:
            m.append(sf.motion.as_array())
        i += 1
    src.release()
    return np.cumsum(np.array(m), axis=0)


def _highpass(x, w=15):
    pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
    return x - sliding_window_view(pad, w, axis=0).mean(axis=-1)


def hf_score(path, cfg, n=300, out_dir="hf"):
    """High-frequency suppression: the metric that matches what the eye sees.

    Strips the low-frequency component (the deliberate camera motion) from
    both input and output, then compares the standard deviation of what is
    left. Returns (x %, y %, saturation %).
    """
    r = stabilize_clip(path, cfg, out_dir=out_dir, max_frames=n,
                       side_by_side=False, to_h264=False, quiet=True)
    z = 1/(1-2*cfg.crop_ratio)
    a = _highpass(_trajectory(path, n))
    b = _highpass(_trajectory(r["out"], n)) / z
    c = cv2.VideoCapture(path)
    W, H = int(c.get(3)), int(c.get(4))
    c.release()
    mx, my = cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95
    co = r["corrections"]
    sat = max((np.abs(co[:, 0]) >= mx*.98).mean(),
              (np.abs(co[:, 1]) >= my*.98).mean())
    return (100*(1-np.std(b[:, 0])/np.std(a[:, 0])),
            100*(1-np.std(b[:, 1])/np.std(a[:, 1])), 100*sat)


def evaluate_gt(gt_path, result, baseline=None, verbose=True):
    """How much of the injected shake came back out.

    `baseline` is the same clip stabilized *without* injected shake. Subtracting
    it removes the source footage's own shake, which otherwise counts as error.
    Note this assumes the system is linear, so it stops being valid once the
    correction saturates -- a clip whose deliberate motion already fills the
    crop budget will produce meaningless (sometimes negative) numbers here.
    """
    gt = np.load(gt_path)
    corr = result["corrections"]
    n = min(len(gt), len(corr))
    gt, corr = gt[:n], corr[:n]
    resid = gt + corr
    if baseline is not None:
        b = baseline["corrections"]
        m = min(n, len(b))
        gt, corr, resid, n = gt[:m], corr[:m], resid[:m] - b[:m], m
    before, after = np.std(gt[:, :2], axis=0), np.std(resid[:, :2], axis=0)
    ba, aa = math.degrees(np.std(gt[:, 2])), math.degrees(np.std(resid[:, 2]))
    out = {"n": n, "baseline": baseline is not None,
           "x_before": float(before[0]), "x_after": float(after[0]),
           "y_before": float(before[1]), "y_after": float(after[1]),
           "a_before": ba, "a_after": aa,
           "x_red": float((1-after[0]/before[0])*100) if before[0] else 0.0,
           "y_red": float((1-after[1]/before[1])*100) if before[1] else 0.0,
           "a_red": float((1-aa/ba)*100) if ba else 0.0,
           "gt": gt, "resid": resid, "corr": corr}
    if verbose:
        tag = "baseline subtracted" if baseline is not None else "no baseline"
        print(f"ground truth, {n} frames, {tag}")
        print(f"{'axis':>6s} {'injected':>10s} {'residual':>10s} {'suppressed':>11s}")
        print(f"{'x':>6s} {out['x_before']:9.2f}px {out['x_after']:9.2f}px {out['x_red']:10.1f}%")
        print(f"{'y':>6s} {out['y_before']:9.2f}px {out['y_after']:9.2f}px {out['y_red']:10.1f}%")
        print(f"{'angle':>6s} {out['a_before']:9.2f}d  {out['a_after']:9.2f}d  {out['a_red']:10.1f}%")
    return out


def show(path, width=960):
    """Play a clip inline. Left half raw, right half stabilized."""
    from base64 import b64encode
    from IPython.display import HTML
    mb = os.path.getsize(path)/1e6
    if mb > 60:
        print(f"warning: {mb:.0f} MB embedded; lower max_frames")
    d = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width={width} controls loop>'
                f'<source src="data:video/mp4;base64,{d}" type="video/mp4"></video>')


print("Tools loaded.")

### A3. Plots and batch runner


In [ ]:
"""Plotting helpers for the notebook."""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


def summary_table(results):
    print(f"{'clip':12s} {'resolution':>11s} {'frames':>7s} {'ms':>7s} {'p95':>7s} "
          f"{'FPS':>6s} {'transl.':>9s} {'rot.':>8s} {'sat_x':>7s} {'sat_y':>7s}")
    print("-" * 92)
    for r in results.values():
        rot = "n/a" if r.get("rot_valid") is False else f"{r['da_red']:7.1f}%"
        print(f"{r['name']:12s} {r['res']:>11s} {r['frames']:7d} {r['ms']:7.2f} "
              f"{r['ms_p95']:7.2f} {r['fps']:6.1f} {r['dxy_red']:8.1f}% "
              f"{rot:>8s} {r['sat_x']:6.1f}% {r['sat_y']:6.1f}%")
    print("-" * 92)
    print("transl./rot. = end-to-end suppression, zoom corrected")
    print("sat_*        = share of frames where the correction hit the crop limit")
    print("rot. = n/a   = clip has < 0.2 deg of rotation; the ratio is noise")


def plot_all(results, cfg):
    names = list(results)
    idx = np.arange(len(names))
    w = .38
    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle(f"alpha={cfg.smooth_alpha}, crop={cfg.crop_ratio}, "
                 f"adaptive={cfg.adaptive}", fontsize=13, fontweight="bold")

    a = ax[0, 0]
    b1 = a.bar(idx-w/2, [results[n]["dxy_red"] for n in names], w,
               label="translation", color="royalblue", alpha=.85)
    b2 = a.bar(idx+w/2, [results[n]["da_red"] for n in names], w,
               label="rotation", color="darkorange", alpha=.85)
    for bs in (b1, b2):
        for bar in bs:
            a.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                   f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8)
    a.set_title("End-to-end suppression"); a.set_ylabel("%")
    a.set_xticks(idx); a.set_xticklabels(names); a.legend(fontsize=8)
    a.grid(alpha=.3, axis="y")

    a = ax[0, 1]
    a.bar(idx-w/2, [results[n]["sat_x"] for n in names], w, label="x",
          color="royalblue", alpha=.85)
    a.bar(idx+w/2, [results[n]["sat_y"] for n in names], w, label="y",
          color="seagreen", alpha=.85)
    a.axhline(10, ls=":", c="red", lw=1.3, label="10% threshold")
    a.set_title("Frames at the crop limit"); a.set_ylabel("% of frames")
    a.set_xticks(idx); a.set_xticklabels(names); a.legend(fontsize=8)
    a.grid(alpha=.3, axis="y")

    a = ax[1, 0]
    for n in names:
        t = results[n]["times"]
        a.plot(np.arange(len(t)), t, lw=.8, alpha=.75, label=n)
    a.axhline(33.33, ls=":", c="red", lw=1.4, label="30 FPS budget")
    a.set_title("Per-frame processing time")
    a.set_xlabel("frame"); a.set_ylabel("ms"); a.legend(fontsize=8); a.grid(alpha=.3)

    a = ax[1, 1]
    for n in names:
        co = results[n]["corrections"]
        a.plot(np.hypot(co[:, 0], co[:, 1]), lw=1.0, alpha=.8, label=n)
    a.set_title("Magnitude of the applied correction")
    a.set_xlabel("frame"); a.set_ylabel("pixels"); a.legend(fontsize=8); a.grid(alpha=.3)

    plt.tight_layout(); plt.show()


def plot_trajectory(path, cfg, max_frames=300):
    """Raw vs smoothed trajectory. The shaded gap is the correction; the dashed
    red lines are the crop limit. If the correction rides the limit, crop_ratio
    is too small for that clip."""
    src = VideoFileSource(path)
    stab = OnlineStabilizer(cfg)
    traj, sm, n = [], [], 0
    while n < max_frames:
        f = src.read()
        if f is None:
            break
        stab.process(f)
        traj.append(stab.traj.copy())
        sm.append(stab.smooth.copy())
        n += 1
    src.release()
    traj, sm = np.array(traj), np.array(sm)
    W, H = stab.size
    lim = [cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95]
    labels = ["x (px)", "y (px)", "angle (deg)"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for i, a in enumerate(axes):
        t = np.degrees(traj[:, i]) if i == 2 else traj[:, i]
        s = np.degrees(sm[:, i]) if i == 2 else sm[:, i]
        a.plot(t, lw=1.0, alpha=.75, label="raw trajectory")
        a.plot(s, lw=1.8, label="smoothed (applied)")
        a.fill_between(range(len(t)), t, s, alpha=.18, label="correction")
        if i < 2:
            a.plot(t+lim[i], "--", lw=.7, c="red", alpha=.5)
            a.plot(t-lim[i], "--", lw=.7, c="red", alpha=.5, label="crop limit")
        a.set_ylabel(labels[i]); a.legend(fontsize=8, loc="upper right"); a.grid(alpha=.3)
    axes[-1].set_xlabel("frame")
    fig.suptitle(f"{os.path.basename(path)} — alpha={cfg.smooth_alpha}, "
                 f"crop={cfg.crop_ratio}")
    plt.tight_layout(); plt.show()


print("Plots loaded.")


"""Batch runner: same config across every clip so comparisons are fair."""

import cv2
import numpy as np


ROT_MIN_DEG = 0.2      # below this the rotation ratio is noise, not signal


def run_all(clips, cfg, max_frames=None):
    z = 1 / (1 - 2*cfg.crop_ratio)
    results = {}
    for name, path in clips.items():
        print(f"[{name}] running...", flush=True)
        r = stabilize_clip(path, cfg, out_dir="out", max_frames=max_frames,
                           side_by_side=True, quiet=True)
        n = r["frames"]
        a = measure_motion(path, n)
        b = measure_motion(r["out"], n, half="right")
        b = {**b, "dxy": b["dxy"] / z}          # output is zoomed
        c = cv2.VideoCapture(path)
        W, H = int(c.get(3)), int(c.get(4))
        c.release()
        mx, my = cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95
        co = r["corrections"]
        r.update({
            "name": name, "in_path": path, "zoom": z, "m_in": a, "m_out": b,
            "dxy_red": (1 - b["dxy"]/a["dxy"])*100 if a["dxy"] else 0.0,
            "da_red": (1 - b["da_std"]/a["da_std"])*100 if a["da_std"] else 0.0,
            "rot_valid": a["da_std"] >= ROT_MIN_DEG,
            "sat_x": float((np.abs(co[:, 0]) >= mx*0.98).mean()*100),
            "sat_y": float((np.abs(co[:, 1]) >= my*0.98).mean()*100),
        })
        results[name] = r
    print("done.")
    return results


print("Runner loaded.")

---
## B. Data and settings

`meshflow` and `ostrich` download automatically. `running.mp4` and any Jetson
recordings you want to test go in the file panel on the left.


In [ ]:
import os

CLIPS = {}
CLIPS["meshflow"] = download_sample("meshflow")   # MeshFlow ECCV 2016 sample
CLIPS["ostrich"] = download_sample("ostrich")     # handheld

for extra in ("running.mp4",):                    # drop your own clips here
    if os.path.exists(extra):
        CLIPS[os.path.splitext(extra)[0]] = extra
    else:
        print(f"note: {extra} not found, skipping")

print()
for n, p in CLIPS.items():
    c = cv2.VideoCapture(p)
    print(f"{n:12s} {int(c.get(3))}x{int(c.get(4))}  {c.get(5):4.1f}fps  "
          f"{int(c.get(7)):5d} frames")
    c.release()

### Settings

One config for every clip, otherwise the comparison isn't fair.

These values come from the sweeps in the appendix. `crop_ratio=0.08` sits at the
knee of the crop-vs-suppression curve: going to 0.10 costs another 4% of the
frame and buys about 2 points. `adaptive` is off because it measured worse
almost everywhere.


In [ ]:
CFG = StabConfig(
    smooth_alpha=0.92,
    crop_ratio=0.08,       # 16% of the frame, zoom 1.19x
    adaptive=False,
    max_angle_deg=3.0,
)
MAX_FRAMES = 300

print(f"alpha={CFG.smooth_alpha}  crop={CFG.crop_ratio}  "
      f"zoom={1/(1-2*CFG.crop_ratio):.3f}  adaptive={CFG.adaptive}")

---
## C. Run


In [ ]:
RESULTS = run_all(CLIPS, CFG, MAX_FRAMES)

---
## D. Results

Two things to watch. `sat_x`/`sat_y` above 10% means the correction is hitting
the crop limit on that clip and `crop_ratio` is the binding constraint, not the
filter. Rotation shows `n/a` when a clip has almost no rotation to begin with --
dividing a small number by a small number just measures noise.


In [ ]:
summary_table(RESULTS)

### Charts


In [ ]:
plot_all(RESULTS, CFG)

### Videos

Left half raw, right half stabilized. The numbers can only take you so far;
this is where the decision actually gets made.


In [ ]:
for n, r in RESULTS.items():
    print(f"\n=== {n}  ({r['res']}, translation -{r['dxy_red']:.0f}%, "
          f"rotation -{r['da_red']:.0f}%) ===")
    display(show(r["out"]))

### One clip in detail


In [ ]:
CLIP = "meshflow"
plot_trajectory(CLIPS[CLIP], CFG, MAX_FRAMES)

---
## E. Deployment

The Jetson only needs section A1. Copy it to `core.py`, pair it with
`run_camera.py` from the repo, and swap `VideoFileSource` for
`CsiCameraSource`.

```bash
gst-launch-1.0 nvarguscamerasrc num-buffers=30 ! fakesink   # camera alive?
python3 record_clips.py --sensor-id 0 --seconds 30          # get real footage
python3 run_camera.py --sensor-id 0 --frames 900 --no-display
```

Measure p95, not the mean: 30 FPS means every frame under 33.3 ms, and an
average of 20 ms with a p95 of 40 ms still drops frames. Close VS Code first,
its server eats a real share of the CPU on an Orin Nano.

If p95 goes over budget, in order: `--proc-width 320`, then `--cuda` for the
warp, then VPI for optical flow.


---
# Appendix — how the settings were chosen

Nothing below runs as part of the main flow. These are the experiments that
produced the defaults, kept because the reasoning matters more than the
numbers and because a few of them found real bugs.

Each section says what it measures and what came out of it.


## A. Which metric to trust

End-to-end suppression counts the deliberate camera motion we're supposed to
keep, so on pan-heavy footage it can't get near 100%. Splitting the motion into
what's intentional and what's shake makes that concrete: on one of the Jetson
recordings 97% of the movement was intentional, which is why 40% looked like a
bad score when it was close to the ceiling.

`hf_score` strips the low-frequency part first. That's the number worth
optimizing, and switching to it changed several conclusions.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def motion_split(path, n=300):
    """Split the trajectory into intentional motion and shake."""
    cum = _trajectory(path, n)
    def lowpass(x, w=31):
        pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
        return sliding_window_view(pad, w, axis=0).mean(axis=-1)
    lp = lowpass(cum); hp = cum - lp
    print(f"{os.path.basename(path)}")
    for i, lbl in enumerate(["x", "y"]):
        low, high = np.std(lp[:, i]), np.std(hp[:, i])
        print(f"  {lbl}: intentional={low:7.1f}px  shake={high:6.1f}px"
              f"  -> shake is {100*high**2/(high**2+low**2):.0f}% of the energy")

for n, p in CLIPS.items():
    motion_split(p, 300)

## B. Crop ratio

Every pixel of crop is field of view given up, so this is the trade-off that
actually needs a decision. The curve has a knee around 0.08: below it the
suppression falls off quickly, above it each extra 4% of the frame buys about
2 points.

Worth showing to whoever owns the FOV budget rather than picking alone.


In [ ]:
for cr in (0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.14):
    row = f"crop={cr:<5} zoom={1/(1-2*cr):.2f}x  loss={200*cr:3.0f}%  "
    for n in ("meshflow", "running"):
        if n not in CLIPS: continue
        x, y, s = hf_score(CLIPS[n], StabConfig(smooth_alpha=0.92, crop_ratio=cr,
                                                adaptive=False, max_angle_deg=3.0))
        row += f"| {n}: x={x:5.1f} y={y:5.1f} sat={s:3.0f}% "
    print(row)

## C. Smoothing strength

Two clips want opposite things. `meshflow` (walking) improves steadily as alpha
goes up. `running` gets worse — the camera moves fast enough that the extra lag
fills the crop budget and the correction saturates.

0.92 is the compromise. On footage with sustained fast motion 0.85 is the better
choice, which is a finding rather than a problem.


In [ ]:
for al in (0.85, 0.92, 0.95):
    row = f"alpha={al}  "
    for n in ("meshflow", "running"):
        if n not in CLIPS: continue
        x, y, s = hf_score(CLIPS[n], StabConfig(smooth_alpha=al, crop_ratio=0.08,
                                                adaptive=False, max_angle_deg=3.0))
        row += f"| {n}: x={x:5.1f} y={y:5.1f} sat={s:3.0f}% "
    print(row)

## D. Other smoothers

An EMA lags behind sustained motion by roughly `v * a/(1-a)`, and that lag eats
crop budget. Several filters that shouldn't have that problem were tried:
alpha-beta and alpha-beta-gamma (track constant velocity/acceleration with no
steady-state error), and cascaded EMAs (sharper cutoff).

None of them beat the plain EMA. The cascades push saturation up because the
extra stages add their own delay. A look-ahead variant was also tested offline
with up to 400 ms of buffering and still didn't win, which is worth reporting on
its own: the real-time constraint isn't costing anything here.


In [ ]:
def set_smoother(kind):
    """kind: 'ema' | 'ab' (constant velocity) | 'ema2' (cascaded)"""
    def _step(self, mx, my):
        cfg = self.cfg; al = cfg.smooth_alpha
        if kind == "ab":
            if not hasattr(self, "_vel"): self._vel = np.zeros(3)
            g = 1-al; bg = g*g/(2-g)
            pred = self.smooth + self._vel; res = self.traj - pred
            self.smooth = pred + g*res; self._vel = self._vel + bg*res
        elif kind == "ema2":
            if not hasattr(self, "_s1"): self._s1 = np.zeros(3)
            self._s1 = al*self._s1 + (1-al)*self.traj
            self.smooth = al*self.smooth + (1-al)*self._s1
        else:
            self.smooth = al*self.smooth + (1-al)*self.traj
        corr = self.smooth - self.traj
        ma = math.radians(cfg.max_angle_deg)
        corr = self._soft_limit(corr, mx, my, ma)
        self.smooth += cfg.windup_leak * ((self.traj + corr) - self.smooth)
        return corr
    OnlineStabilizer._smooth_step = _step

P = CLIPS["meshflow"]
for kind in ("ema", "ab", "ema2"):
    for al in (0.85, 0.92):
        set_smoother(kind)
        x, y, s = hf_score(P, StabConfig(smooth_alpha=al, crop_ratio=0.08,
                                         adaptive=False, max_angle_deg=3.0))
        print(f"{kind:5s} alpha={al}  x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%")
set_smoother("ema")   # back to default

## E. Ground truth

Inject shake of a known strength, then check how much came back out. This is the
cleanest signal that the core is correct — it's how the rotation sign bug and
the zoom scaling bug were both caught.

One caveat: subtracting the baseline assumes the system is linear, so it breaks
once the correction saturates. A clip whose own motion already fills the crop
budget will produce meaningless (sometimes negative) numbers here.


In [ ]:
P = CLIPS["meshflow"]
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.10,
                 adaptive=False, max_angle_deg=6.0)

shaken, gt_path = add_shake(P, amp=15.0, rot=2.0, max_frames=300)
base = stabilize_clip(P, cfg, out_dir="gt_base", max_frames=300,
                      side_by_side=False, to_h264=False, quiet=True)
res = stabilize_clip(shaken, cfg, max_frames=300, quiet=True)
evaluate_gt(gt_path, res, baseline=base)

### Rotation on its own

Rotation suppression sits below translation, so it was isolated with a pure
rotation injection (`amp=0`). It came out around 73–74% on both clips, which
rules out a sign or composition error in the warp — it's just the ceiling.


In [ ]:
for clip in ("meshflow", "ostrich"):
    p = CLIPS[clip]
    sh, gtp = add_shake(p, amp=0.0, rot=3.0, max_frames=250)
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.10,
                     adaptive=False, max_angle_deg=6.0)
    base = stabilize_clip(p, cfg, out_dir=f"pr_{clip}", max_frames=250,
                          side_by_side=False, to_h264=False, quiet=True)
    r = stabilize_clip(sh, cfg, out_dir=f"pr2_{clip}", max_frames=250,
                       side_by_side=False, to_h264=False, quiet=True)
    ev = evaluate_gt(gtp, r, baseline=base, verbose=False)
    print(f"{clip:10s} pure rotation suppressed: {ev['a_red']:.1f}%")

## F. Where the remaining shake comes from

This one explains the ceiling. Split the frame into quadrants and measure each
one separately: if the quadrants disagree more than they agree, the leftover
motion isn't global and no single affine transform can remove it — fixing one
corner necessarily breaks another.

On the Jetson recordings the disagreement was larger than the common motion
(116% on input, 131% on output). That's rolling shutter, parallax, or both, and
it's the reason parameter tuning stopped helping. Getting past it means a
spatially-varying model such as MeshFlow, not a better filter.


In [ ]:
def quadrant_test(path, cfg, n=200):
    r = stabilize_clip(path, cfg, out_dir="quad", max_frames=n,
                       side_by_side=False, to_h264=False, quiet=True)
    for label, vid in [("input", path), ("output", r["out"])]:
        src = VideoFileSource(vid)
        sts = [OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
               for _ in range(4)]
        acc = [[] for _ in range(4)]
        i = 0
        while i < n:
            f = src.read()
            if f is None: break
            h, w = f.image.shape[:2]
            quads = [(0,h//2,0,w//2), (0,h//2,w//2,w),
                     (h//2,h,0,w//2), (h//2,h,w//2,w)]
            for k, (y0,y1,x0,x1) in enumerate(quads):
                sub = Frame(f.image[y0:y1, x0:x1], f.timestamp, f.index)
                sf = sts[k].process(sub)
                if sf.motion.valid: acc[k].append(sf.motion.as_array()[:2])
            i += 1
        src.release()
        A = [np.array(a) for a in acc]
        m = min(len(a) for a in A)
        A = np.stack([a[:m] for a in A])
        common = A.mean(axis=0)
        spread = np.linalg.norm(A - common, axis=2).mean()
        mag = np.linalg.norm(common, axis=1).mean()
        print(f"{label:7s} common={mag:6.2f}px  spread={spread:6.2f}px  "
              f"ratio={100*spread/max(mag,1e-6):.0f}%")

quadrant_test(CLIPS["meshflow"], CFG)

## G. Bugs this process caught

Worth writing down, because each was found by a measurement rather than by
reading the code, and each moved the numbers.

**Rotation sign.** `estimateAffinePartial2D` and `getRotationMatrix2D` disagree
on which way positive rotation goes. Measuring the angle and applying it back
with the same sign roughly doubled the rotation. A/B test on a synthetic clip:
input 2.64°, output 4.93° before the fix, 0.67° after.

**Zoom scaling.** The crop-zoom magnifies the image by z, so it magnifies the
motion inside it too, but the correction wasn't scaled to match. A fraction
`(z-1)/z` of every shake survived — 32% at crop=0.12. Controlled test: 28.4 px
input, 10.3 px output before, 2.1 px after.

**Rotation budget.** Translation was limited per-axis, which ignores the extra
margin rotation eats at the corners. At crop=0.10 with 6° of rotation the frame
overflowed by 53 px and `BORDER_REPLICATE` smeared the edges — hard to name,
easy to see. Now all four corners are checked against the combined transform.

**Measurement bias.** `compare_motion` compared an unzoomed input against a
zoomed output, so results got *worse* as crop_ratio went up. Correcting for zoom
flattened it: what looked like 51%→29% across crop 0.06–0.20 was 57% throughout.

**Metric choice.** End-to-end motion rewards lag, which made a look-ahead
variant look worse than the causal one and pushed alpha too high. Switching to
high-frequency suppression reversed several conclusions.
